# Stage 3.5 — INLP / LEACE causal intervention sweep

For each model in the probing subset, fit linear-concept-erasure projections (INLP + LEACE) at the peak-probe layer per attribute, sanity-check, then re-run CrowS-Pairs and StereoSet with the projection hook attached.

**Prereq:** the standard logit + probing sweeps must have already produced `results/activations/<model>/` and `results/probe_results/<model>/<attr>.json` for the probing-subset models.

**Workflow:** validate first on one model → inspect → then commit to the full sweep.

In [ ]:
GITHUB_REPO = 'https://github.com/korneli777/llm-bias-eval.git'
DRIVE_DIR = '/content/drive/MyDrive/llm-bias-eval'

In [ ]:
import os, subprocess, shutil
from google.colab import drive, userdata
from huggingface_hub import login

REPO_DIR = '/content/llm-bias-eval'

drive.mount('/content/drive')
os.makedirs(f'{DRIVE_DIR}/results', exist_ok=True)

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', GITHUB_REPO, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(['git', 'pull', '--ff-only'], check=True)

results_link = os.path.join(REPO_DIR, 'results')
if os.path.lexists(results_link):
    (os.unlink if os.path.islink(results_link) else shutil.rmtree)(results_link)
os.symlink(f'{DRIVE_DIR}/results', results_link)

subprocess.run(['pip', 'install', '-q', '-e', REPO_DIR], check=True)

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
os.environ['WANDB_PROJECT'] = 'llm-bias-eval'
login(token=os.environ['HF_TOKEN'])

## Step 1 — Validate on Llama-3.1-8B (one model, gender, both methods)

`--validate-only` fits the projections + runs both sanity checks (probe nullification, LM perplexity blow-up) but **does not** re-run benchmarks. Inspect the sanity report under `results/intervention/<model>/<attr>__<method>__sanity.json` before committing to the full sweep.

Pass criteria:
- `nullification.passed = True` (post-projection probe accuracy ≤ 0.55)
- `perplexity.passed = True` (post/pre ratio ≤ 1.5×)

If either fails, adjust `--max-iter` (INLP) or pick a different intervention layer before scaling up.

In [ ]:
!python scripts/run_intervention.py \
    --models meta-llama/Llama-3.1-8B \
    --attributes gender \
    --methods inlp leace \
    --validate-only \
    --no-wandb

In [ ]:
# Inspect the sanity report inline.
import json
from pathlib import Path
for fp in sorted(Path('results/intervention/meta-llama__Llama-3.1-8B').glob('*__sanity.json')):
    print('===', fp.name, '===')
    print(json.dumps(json.load(open(fp)), indent=2)[:1500])
    print()

## Step 2 — Full sweep (only after validation passes)

12 probe-subset models × 2 attributes (gender, race) × 2 benchmarks (CrowS-Pairs, StereoSet) × 2 prompt_modes (raw, instruct) × 2 methods (INLP, LEACE) = 192 cells.

Resumable per cell — `is_completed()` skips finished JSONs. Estimated ~10 hours total on H100; you can safely interrupt and re-run.

In [ ]:
!python scripts/run_intervention.py \
    --benchmarks crows_pairs stereoset \
    --attributes gender race \
    --methods inlp leace \
    --prompt-modes raw instruct